# Oppitunti 18: AI-tekijöiden suojaaminen kryptografisilla kuiteilla

## Käytännön harjoitusvihko

Tämä vihko opastaa läpi neljän harjoituksen:

1. **Allekirjoita ensimmäinen kuitteesi** agenttityökalukutsulle ja varmista se.
2. **Muokkaa kuittia väärin** ja katso varmennuksen epäonnistuminen.
3. **Rakenna kolmen kuitin ketju** ja varmista ketjun eheys.
4. **Kääri Microsoft Agent Framework -työkalukutsu** niin, että jokainen toiminto tuottaa kuitin.

Kaikki kryptografiset perusoperaatiot on tuotua hyvin ylläpidetyistä kirjastoista (`pynacl` Ed25519:lle, `jcs` RFC 8785 -kannoniselle JSONille, `hashlib` Pythonin standardikirjastosta SHA-256:lle). Kuittilogiikka itse on tavallista Pythonia, jota voit lukea ja muokata.

Suorita solut järjestyksessä. Jokainen osio on lyhyt ja itsemääräytyvä.


## Asennus

Asenna kaksi riippuvuutta. Molemmilla on sallivat lisenssit (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Apuohjelmat

Nämä kaksi apuohjelmaa käsittelevät base64url-koodausta (ilman täytemerkkejä) ja SHA-256-tiivisteen luomista mielivaltaisista objekteista. Ne pitävät muunkin muistiinpanon keskittyneenä kuittilogiikkaan.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Osa 1: Allekirjoita ensimmäinen kuittisi

Kuvittele, että **Contoso Travel** -agenttimme juuri etsi lentoja Sydneyistä Los Angelesiin asiakkaalle. Haluamme tallentaa portin lausunnon tästä työkalukutsusta, jotta tuleva tarkastaja voi varmistaa sen erikseen kiinnitetyn myöntäjän avaimen alla.

### Vaihe 1.1: Luo allekirjoitusavain

Tuotantoympäristössä agentin allekirjoitusavain sijaitsee laitteistoturvamoduulissa (HSM), Azure Key Vaultissa tai vastaavassa suojatussa varastossa. Tässä oppitunnissa luomme uuden avaimen muistiin.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Vaihe 1.2: Rakenna kuittipaketti

Paketti sisältää kaiken, mitä haluamme kuitin todistavan: kuka toimi, mitä työkalua käyttäen, millä argumenteilla, mitä palautui, minkä politiikan alaisena ja milloin. Tiivistämme argumentit ja tuloksen emmekä sisällytä niitä suoraan, jotta kuitti ei vuoda arkaluontoista sisältöä.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Vaihe 1.3: Allekirjoita ja koosta kuitti

Kolme vaihetta:

1. Canonicalisoi hyötykuorma JCS:n avulla niin, että kaksi saman loogisen kuitin tuottavaa toteutusta tuottavat identtiset tavujonot.
2. Allekirjoita canonicalisoinnin tavut suoraan Ed25519-yksityisavaimella. PureEdDSA hajauttaa viestin sisäisesti, joten ylimääräinen esihajaus muuttaisi protokollaa.

Allekirjoitus liitetään sitten alkuperäiseen hyötykuormaan lopullisen kuitin tuottamiseksi.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Vaihe 1.4: Tarkista kuitti

Varmistus kääntää prosessin. Poistamme allekirjoituksen, laskemme kanoniset tavut uudelleen ja tarkistamme allekirjoituksen julkisen avaimen perusteella.

Varmistus on offline-tilassa: ei palvelupyyntöjä eikä verkkoyhteyttä. Se tarvitsee kuitenkin yhden asian kuitin lisäksi: allekirjoittajan julkisen avaimen, joka saadaan kanavaa pitkin, johon tarkastaja jo luottaa, kuten JWK-joukosta, jonka allekirjoittaja julkaisee, tai etukäteen asetetusta arvosta. `public_key`-kenttä kuitin sisällä ei voi toimia tuossa roolissa, koska kuka tahansa, joka kirjoittaa kuitin, kirjoittaa myös sen kentän. Osa 2 näyttää, millaista väärennöstä tämä estää.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Sinun pitäisi nähdä `Receipt is valid: True`, `Pre-hashed receipt valid: False` ja `Malformed receipts rejected: True`. Positiivinen tapaus osoittaa, että suora-JCS-allekirjoituspolku toimii; negatiivinen kontrolli tekee allekirjoitusalueen säännöstä suoritettavan sen sijaan, että se jäisi proosamuotoiseksi; viimeinen rivi näyttää virheellisesti muotoillun syötteen palauttavan `False` sen sijaan, että se aiheuttaisi poikkeuksen.


## Osa 2: Muokkaa kuittia

Kuittien koko idea on, että ne ovat muokkausturvallisia. Todistetaan se.

Muokkaamme täsmälleen yhtä merkkiä kuitissa ja katsomme, että varmistus epäonnistuu.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Mitä juuri tapahtui?

Kun muutimme `policy_id`, kanoniset tavut muuttuivat. Allekirjoitus (joka oli alkuperäisten kanonisten tavujen päällä) ei enää täsmää. Varmennus palauttaa oikein `False`.

Alle kirjoitetun kentän, eli mitä tahansa `signature` -objektin ulkopuolella, muuttaminen ja silti saada kuitti vahvistumaan julkisen avaimen kanssa ei ole mahdollista, ellei hyökkääjällä ole julkaisijan yksityistä avainta. `signature` -objekti ei itsessään ole allekirjoitettu: `sig` on allekirjoitus, ja `verify_receipt` jättää `public_key` huomioimatta, joten pelkkä tuon kentän muokkaus ei muuta mitään. Kunhan yksityinen avain on avainholvissa ja varmistajat käyttävät julkaisijan itsenäisesti julkaistua avainta, allekirjoitetun kentän muuttaminen on mahdotonta peittää. Tämä ehto on tärkeä: seuraava solu näyttää muokatun kuitin, joka varmistuu täydellisesti toista avainta vastaan.

Kokeile itse: muokkaa yllä olevan solun `tool_name`, `agent_id` tai `timestamp` ja suorita uudelleen. Jokainen muutos allekirjoitettuun kenttään tuottaa virheellisen kuitin.


### Väärennös, jonka allekirjoitus ei paljasta

Edellä mainittu hyökkääjä ei allekirjoittanut uudelleen. Tarkempi tekijä tekisi näin: muuttaisi kuitin,
allekirjoittaisi sen itse generoimallaan avaimella ja laittaisi kyseisen avaimen
`public_key` -kenttään. Tuloksena on kuitti, jonka allekirjoitus on täysin aito
sen sisällön osalta.

Suorita tämä ja katso, kumpi kahdesta tarkistuksesta sen löytää.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Sinun pitäisi nähdä `True` ja sitten `False`. Kolme `assert` riviä tekevät siitä
tarkistuksen: solu nostaa virheen, jos väärennös joskus varmistuu myöntäjän avaimella.


Väärennös on sisäisesti yhdenmukainen. Sen allekirjoitus todella vastaa sen
sisältöä ja kantamaa avainta, joten vahvistaja, joka lukee avaimen kuitista,
raportoi sen päteväksi. Kuitissa ei näytä olevan mitään vikaa,
koska siinä ei ole ristiriitoja.

Ainoa asia, joka sen havaittaa, on tarkistus avainta vastaan, johon hyökkääjällä ei ollut
mahdollisuutta vaikuttaa. Tästä syystä `verify_receipt` ottaa avaimen argumenttina: 
funktiota, jonka allekirjoitus on `verify_receipt(receipt)`, ei voi kirjoittaa turvallisesti,
koska ainoa avain, johon se voisi päästä, on hyökkääjän antama.

Sama kuitti tallennetaan nimellä `sample_receipts/04_resigned_receipt.json`, jos haluat
tarkastella sitä notebookin ulkopuolella.


## Osa 3: Ketjuta kuitit yhteen

Yksi kuitti suojaa yhtä toimintoa. Useimmat edustajat suorittavat monia toimintoja. Jotta koko sekvenssi olisi väärentämisen havaitseva, linkitämme jokaisen kuitin edelliseen lisäämällä edellisen kuitin tiiviste uuden kuitin hyötykuormaan.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Jos joku poistaa tai muuttaa kuittien järjestystä esitetyn sekvenssin sisällä, seuraava linkki katkeaa. Leikkausten havaitseminen tai allekirjoittajan hallitseman loppuosan uudelleenkirjoittamisen havaitseminen vaatii myös kiinnitetyn nykyisen pään, ulkoisen tarkistuspisteen tai todennetun väestölupauksen.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Nyt katkaise ketju manipuloimalla keskimmäistä kuittiä ja varmista uudelleen. Manipuloitu kuitti epäonnistuu allekirjoituksen tarkistuksessa, JA seuraava kuitti epäonnistuu ketjulinkkitarkistuksessa (koska sen `previous_receipt_hash` ei enää vastaa muokatun keskimmäisen kuitin hajautusarvoa).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Kuitti 0 tarkistuu edelleen (sitä ei muokattu, eikä sillä ole edeltäjää, johon se voisi nojata). Kuitti 1 epäonnistuu allekirjoituksen tarkistuksessa, koska muutimme `tool_args_hash`. Kuitti 2 epäonnistuu ketjun linkin tarkistuksessa, koska sen `previous_receipt_hash` laskettiin alkuperäistä (nyt muokattua) kuittia 1 vastaan.

Vaikka hyökkääjä allekirjoittaisi uudelleen muokatun kuitin 1, uusi allekirjoitus ei onnistu antajan avaimella, ellei hyökkääjällä ole antajan yksityisavainta (oman avaimensa uudelleenkirjoitus epäonnistuu, kuten kappale 2 osoitti), ja ketjun linkin ristiriita kuitissa 2 paljastaisi edelleen manipuloinnin. Muutoksen piilottamiseksi hyökkääjän pitäisi allekirjoittaa uudelleen jokainen kuitti muutoskohdasta eteenpäin, mikä vaatii antajan yksityisavaimen hallussapitoa.


## Osa 4: Kääri agenttityökalun kutsu kuittauksen allekirjoituksella

Todellisessa käyttöönotossa et halua, että jokainen agentin kirjoittaja muistaa kutsua `make_receipt`. Haluat, että kuittauksen allekirjoitus tapahtuu automaattisesti jokaiselle työkalun kutsulle.

Tässä on yksinkertaisin malli: kääreluokka, joka ottaa minkä tahansa kutsuttavan työkalufunktion ja palauttaa siitä kuitin lähettävän version. Tämä mukautuu mihin tahansa agenttikehykseen, mukaan lukien Microsoft Agent Framework (`agent_framework.foundry`).

Jos sinulla ei ole Microsoft Foundry -projektia valmiina, alla oleva paikallinen mock aukenee silti mallin.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integrointi Microsoft Agent Frameworkin kanssa

Yllä oleva `ReceiptedTool`-kääre on kehysriippumaton. Jotta voit käyttää sitä Microsoft Agent Frameworkilla rakennetun agentin sisällä, rekisteröi kääritty funktio työkaluna. Luonnos (korvaisit mallin oikealla Microsoft Foundry -työkalun rekisteröinnillä):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Agenttikehyksen ei tarvitse tietää mitään kuiteista. Kuitin allekirjoittaminen on kääritty työkalun ympärille, ei liitetty kehykseen. Näin lisäät alkuperäisyyden olemassa olevaan agenttikoodiin ilman, että kirjoitat agenttia uudelleen.


## Kertaus ja haaste

Sinulla on:

- Luotu Ed25519-avaimet.
- Rakennettu ja allekirjoitettu kuitti agenttityökalukutsulle.
- Varmistettu kuitti offline-tilassa käyttämällä ainoastaan myöntäjän julkista avainta, hankittu riippumattomasti kuitista.
- Muokattu kuittia ja havaittu varmennuksen epäonnistuminen.
- Rakennettu kolmen kuitin hajautettu ketju.
- Muokattu ketjun keskiosaa ja havaittu sekä allekirjoituksen että ketjun linkin epäonnistuminen.
- Kääritty työkalufunktio automaattisella kuittien allekirjoituksella.

**Haaste.** Laajenna kuittien kaavaa `request_id`-kentällä (UUID hajautettuun jäljitykseen). Päivitä `make_receipt` sisällyttämään tämä kenttä, ja varmista, että kuitit edelleen varmistuvat päästä päähän. Muuta sitten kenttää allekirjoituksen jälkeen ja varmista, että varmennus epäonnistuu. Tämä pakottaa sinut sisäistämään, miten jokainen kanonisen koodauksen tavu vaikuttaa allekirjoitukseen.

**Tärkeä raja.** Pinned-avain- ja ketjukontekstissa kuitit todistavat allekirjoituksen kuuluvuuden, sisällön eheyden ja linkin esitettyyn edeltävään kuitin. Ne eivät todista suoritusta, lähdetotuutta, luotettua aikaa, täydellisyyttä, politiikan arviointia tai oikeellisuutta. Kuitit ovat perustana. Hallinto on järjestelmä, jonka rakennat niiden päälle.

Lue oppitunnin README uudelleen tämän rajan mielessä. Usein tiimien yleisin virhe kuittien kanssa on olettamus, että "meillä on kuitit" tarkoittaa "meitä hallitaan." Se ei tarkoita. Kuitit tekevät agenttitoiminnasta auditoitavaa. Ne eivät tee siitä oikeaa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
